In [1]:
# Baseline: pretrained BART (no fine-tuning)

import os, json
from google.colab import drive

# 1) Mount Drive (this is how we "connect" to Notebook 1 outputs)
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/abstractive_summarizer_bart_lora"
CONFIG_PATH = f"{DRIVE_DIR}/project_config.json"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Config not found at: {CONFIG_PATH}. Did you run Notebook 1 fully?")

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

print("✅ Loaded config:")
print(json.dumps(cfg, indent=2)[:900], "...\n")

Mounted at /content/drive
✅ Loaded config:
{
  "project_name": "abstractive_summarizer_bart_lora",
  "created_utc": "2026-03-04T03:59:32.814999+00:00",
  "seed": 42,
  "dataset": {
    "id": "abisee/cnn_dailymail",
    "config": "3.0.0",
    "splits": [
      "train",
      "validation",
      "test"
    ],
    "text_field": "article",
    "summary_field": "highlights"
  },
  "model": {
    "base_model_id": "facebook/bart-large-cnn",
    "tokenizer_id": "facebook/bart-large-cnn"
  },
  "tokenization": {
    "max_input_tokens": 1024,
    "max_target_tokens": 128,
    "padding": "max_length",
    "truncation": true
  },
  "data_sanity": {
    "min_article_chars": 200,
    "min_summary_chars": 30,
    "scan_sample_n": 5000
  },
  "notes": [
    "Mapping: article -> highlights",
    "Truncation chosen from token-length EDA sample"
  ]
} ...



In [2]:
import random
import numpy as np
import torch

SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seeds set:", SEED)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Seeds set: 42
CUDA available: False


In [3]:
from datasets import load_dataset

DATASET_ID = cfg["dataset"]["id"]
DATASET_CONFIG = cfg["dataset"]["config"]
TEXT_FIELD = cfg["dataset"]["text_field"]
SUMMARY_FIELD = cfg["dataset"]["summary_field"]

ds = load_dataset(DATASET_ID, DATASET_CONFIG)

print(ds)
print("Train columns:", ds["train"].column_names)

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})
Train columns: ['article', 'highlights', 'id']


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BASE_MODEL_ID = cfg["model"]["base_model_id"]
MAX_INPUT_TOKENS = cfg["tokenization"]["max_input_tokens"]
MAX_TARGET_TOKENS = cfg["tokenization"]["max_target_tokens"]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("✅ Loaded model:", BASE_MODEL_ID)
print("Device:", device)
print("Max input tokens:", MAX_INPUT_TOKENS, "| Max target tokens:", MAX_TARGET_TOKENS)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

✅ Loaded model: facebook/bart-large-cnn
Device: cpu
Max input tokens: 1024 | Max target tokens: 128


In [8]:
import torch

#I couldn't find summarization in the pipeline so I'm defining the function myself based on the underlying process the pipelines use anyway.

def summarize_text(text: str,
                   max_input_tokens: int,
                   max_new_tokens: int,
                   num_beams: int = 4,
                   min_new_tokens: int = 30) -> str:
    # Tokenize input (truncate to our decided max input length)
    inputs = tokenizer(
        text,
        max_length=max_input_tokens,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
                  **inputs,
                  max_new_tokens=max_new_tokens,
                  min_new_tokens=min_new_tokens,
                  num_beams=num_beams,
                  length_penalty=2.0,
                  no_repeat_ngram_size=3,
                  early_stopping=True
              )

    # Decode generated ids into text summary
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Now our baseline check

idxs = [0, 25, 100]
for idx in idxs:
    article = ds["validation"][idx][TEXT_FIELD]
    ref = ds["validation"][idx][SUMMARY_FIELD]

    pred = summarize_text(
        article,
        max_input_tokens=MAX_INPUT_TOKENS,
        max_new_tokens=MAX_TARGET_TOKENS,
        num_beams=4
    )

    print("\n==============================")
    print(f"Validation idx={idx}")
    print("REF SUMMARY:\n", ref[:500])
    print("\nBASELINE BART SUMMARY:\n", pred[:500])


Validation idx=0
REF SUMMARY:
 Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

BASELINE BART SUMMARY:
 Zully Broussard gave one of her kidneys to a stranger. Her generosity paired up with big data. It resulted in six patients receiving transplants. Such long-chain transplanting is rare.

Validation idx=25
REF SUMMARY:
 A wag decided to sell "air from Kanye West concert" on eBay .
Bidding got to more than $60,000 before plug was pulled .
Others are now doing similar auctions .

BASELINE BART SUMMARY:
 Kanye West fans are used to seeing thousand-dollar price tags attached to his sneakers. eBay is now being overrun by sellers offering up plastic bags full of air. The gag started Friday when one seller attempted to sell a Zipperseal bag with "Air From Kanye Show"

Validation idx=100
REF SUMMARY:
 Google CFO Patrick Pichette's memo announcing his resignation in order to seek work/life balance we

In [10]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [13]:
import evaluate
from tqdm.auto import tqdm

rouge = evaluate.load("rouge")

EVAL_SPLIT = "validation"
EVAL_N = 30  # start small for speed; knowing that I can increase later
data = ds[EVAL_SPLIT].select(range(min(EVAL_N, len(ds[EVAL_SPLIT]))))

preds, refs = [], []

for ex in tqdm(data, total=len(data)):
    article = ex[TEXT_FIELD]
    reference = ex[SUMMARY_FIELD]

    pred = summarize_text(
        article,
        max_input_tokens=MAX_INPUT_TOKENS,
        max_new_tokens=MAX_TARGET_TOKENS,
        num_beams=4
    )

    preds.append(pred)
    refs.append(reference)

scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
scores

  0%|          | 0/30 [00:00<?, ?it/s]

{'rouge1': np.float64(0.41229190017864614),
 'rouge2': np.float64(0.2090988332596613),
 'rougeL': np.float64(0.3174396785028656),
 'rougeLsum': np.float64(0.3579627329287859)}

### Notebook: Baseline BART Evaluation

In this notebook I evaluated a pretrained summarization model to establish a baseline before applying LoRA fine-tuning.

- Loaded the shared project configuration file created in the previous notebook to ensure consistent dataset, model, and tokenization settings.
- Loaded the CNN/DailyMail dataset and initialized the pretrained model **facebook/bart-large-cnn** as the baseline summarizer.
- Implemented a custom summarization function using `model.generate()` to produce summaries from the validation split.
- Generated baseline summaries for a small subset of the validation dataset to confirm that the inference pipeline works correctly.
- Computed **ROUGE evaluation metrics** on a sampled subset of the validation data to measure baseline summarization performance.
- Verified that the pretrained BART model produces reasonable summaries and established initial ROUGE scores that will later be compared against the LoRA fine-tuned model.

### Baseline ROUGE (sample validation subset)

- ROUGE-1: ~0.41  
- ROUGE-2: ~0.21  
- ROUGE-L: ~0.31  
- ROUGE-Lsum: ~0.35  

These values were computed on a small validation sample to verify the evaluation pipeline before running larger experiments.